In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"

In [6]:
from concept_abstraction.selection import greedy_selection_supervised, lp_selection_supervised, multiple_selection_supervised, greedy_selection_supervised
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
from collections import Counter
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import torch.nn as nn
from torchvision import models
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
import pickle 

In [7]:
is_jupyter = 'ipykernel' in sys.modules

In [8]:
if is_jupyter: 
    seed        = 42
    num_concepts_selected = 112
    out_folder = "cub"
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    num_concepts_selected = args.num_concepts_selected
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [9]:
results = {}
results['parameters'] = {'seed'      : seed,
        'num_concepts_selected': num_concepts_selected,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'num_concepts_selected': 112}


In [10]:
np.random.seed(seed)
random.seed(seed)

In [11]:
def get_performance(selected_concepts,accuracy_by_concept):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
    )

    # Train the model
    mlp.fit(train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

## Perfrect Concepts

In [12]:
results['perfect'] = {}

In [13]:
train = pickle.load(open("../../data/cub/train.pkl","rb"))
test = pickle.load(open("../../data/cub/test.pkl","rb"))

In [14]:
train_X = np.array([i['attribute_label'] for i in train])
train_Y = np.array([i['class_label'] for i in train])
test_X = np.array([i['attribute_label'] for i in test])
test_Y = np.array([i['class_label'] for i in test])

In [16]:
results['perfect']['lp'] = {}
for c in [112]:
    lp_concept_list = lp_selection_supervised(train_X,train_Y,c)
    results['perfect']['lp'][c] = {
        'reward': get_performance(lp_concept_list,np.ones(312)),
        'concepts': lp_concept_list
    }
print("Finished LP")

Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17
Finished LP


In [23]:
manually_selected_concepts = open("../../data/cub/manual_concepts.txt").read().strip().split("\n")
manually_selected_concepts = [int(i) for i in manually_selected_concepts]
results['perfect']['manual'] = {
    'reward': get_performance(manually_selected_concepts,np.ones(312)),
    'concepts': manually_selected_concepts
}
print("Manual Performance {}".format(results['perfect']['manual']['reward']))

Manual Performance 1.0


#### Imperfect Concepts

In [63]:
def sigmoid(z):
    return 1/(1 + np.exp(-z))

train = pickle.load(open("../../data/cub/train_error.pkl","rb"))
test = pickle.load(open("../../data/cub/test_error.pkl","rb"))

pred_train_X = sigmoid(np.array([i['attribute_label'] for i in train]))
train_Y = np.array([i['class_label'] for i in train])
pred_test_X = sigmoid(np.array([i['attribute_label'] for i in test]))
test_Y = np.array([i['class_label'] for i in test])

In [64]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128),
    activation='relu',
    solver='adam',
)
# Train the model
mlp.fit(train_X[:,manually_selected_concepts], train_Y)


# Predict on the test set
y_pred = mlp.predict(pred_test_X[:,manually_selected_concepts])

# Compute accuracy
acc = accuracy_score(test_Y.reshape(-1,1), y_pred)

In [65]:
acc

0.6456679323438039

In [68]:
from sklearn.metrics import f1_score

In [76]:
f1_score(pred_train_X[:,manually_selected_concepts].round().astype(np.int8).flatten().tolist(),train_X[:,manually_selected_concepts].astype(np.int8).flatten())

0.9410007883875157

In [77]:
manually_selected_concepts

[1,
 4,
 6,
 7,
 10,
 14,
 15,
 20,
 21,
 23,
 25,
 29,
 30,
 35,
 36,
 38,
 40,
 44,
 45,
 50,
 51,
 53,
 54,
 56,
 57,
 59,
 63,
 64,
 69,
 70,
 72,
 75,
 80,
 84,
 90,
 91,
 93,
 99,
 101,
 106,
 110,
 111,
 116,
 117,
 119,
 125,
 126,
 131,
 132,
 134,
 145,
 149,
 151,
 152,
 153,
 157,
 158,
 163,
 164,
 168,
 172,
 178,
 179,
 181,
 183,
 187,
 188,
 193,
 194,
 196,
 198,
 202,
 203,
 208,
 209,
 211,
 212,
 213,
 218,
 220,
 221,
 225,
 235,
 236,
 238,
 239,
 240,
 242,
 243,
 244,
 249,
 253,
 254,
 259,
 260,
 262,
 268,
 274,
 277,
 283,
 289,
 292,
 293,
 294,
 298,
 299,
 304,
 305,
 308,
 309,
 310,
 311]

In [45]:
np.max(pred_train_X*(1-pred_train_X))

0.0

In [39]:
train_concept_accuracy = np.mean(pred_train_X == train_X,axis=0)
test_concept_accuracy = np.mean(pred_test_X == test_X,axis=0)

In [19]:
results['imperfect'] = {}

In [20]:
manually_selected_concepts = open("../../data/cub/manual_concepts.txt").read().strip().split("\n")
manually_selected_concepts = [int(i) for i in manually_selected_concepts]


In [21]:
results['imperfect']['manual'] = {'reward': get_performance_real(manually_selected_concepts), 'concepts': manually_selected_concepts}

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [20]:
results['imperfect']['lp'] = {}
for c in results['perfect']['lp']:
    results['imperfect']['lp'][c] = {
        'reward': get_performance_real(results['perfect']['lp'][c]['concepts']),
        'concepts': c
    }

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [21]:
results['imperfect']['multiple'] = {}
for c in results['perfect']['lp']:
    imperfect_concepts = multiple_selection_supervised(train_X,train_Y,c)
    results['imperfect']['multiple'][c] = {
        'reward': get_performance_real(imperfect_concepts), 
        'concepts': imperfect_concepts
    }
results['imperfect']['multiple']

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


{20: {'reward': 0.5338280980324474,
  'concepts': [6,
   20,
   35,
   51,
   54,
   90,
   117,
   132,
   149,
   151,
   163,
   178,
   209,
   212,
   218,
   235,
   236,
   240,
   289,
   304]}}

In [23]:
results['imperfect']['random'] = {}

for c in results['perfect']['lp']:
    random_concepts = random.sample(list(range(312)),c)
    results['imperfect']['random'][c] = {
        'reward': get_performance_real(random_concepts), 
        'concepts': random_concepts
    }
results['imperfect']['random']

{20: {'reward': 0.11529168104936141,
  'concepts': [57,
   12,
   140,
   125,
   114,
   71,
   52,
   279,
   44,
   302,
   216,
   16,
   15,
   47,
   111,
   119,
   258,
   308,
   13,
   287]}}

In [24]:
results['imperfect']['greedy'] = {}

for c in results['perfect']['lp']:
    greedy_concepts = greedy_selection_supervised(train_X,train_Y,c)
    results['imperfect']['greedy'][c] = {
        'reward': get_performance_real(greedy_concepts), 
        'concepts': greedy_concepts
    }
results['imperfect']['greedy']

[0.24998083 0.24995266 0.24560361 0.24473498 0.24395134 0.24265612
 0.24214729 0.241924   0.2411571  0.2411571  0.23769547 0.2339816
 0.23339578 0.2330718  0.23246971 0.22802237 0.22429046 0.22422355
 0.2238877  0.21352187] 0.0543086312740497


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


{20: {'reward': 0.5352088367276493,
  'concepts': [236,
   289,
   240,
   51,
   212,
   20,
   35,
   209,
   235,
   6,
   178,
   218,
   132,
   117,
   151,
   54,
   163,
   149,
   304,
   90]}}

## Intervention

In [22]:
test = pickle.load(open("../../data/cub/test_error.pkl","rb"))


In [24]:
num_concepts = len(manually_selected_concepts)
lp_selection  = lp_selection_supervised(train_X,train_Y,num_concepts)
multiple_selection  = multiple_selection_supervised(train_X,train_Y,num_concepts)
greedy_selection = greedy_selection_supervised(train_X,train_Y,num_concepts)
manual_selection = manually_selected_concepts
random_selection = random.sample(list(range(312)),num_concepts)
accuracies = np.mean(test_X == pred_test_X,axis=0)
imperfect_selection = imperfect_lp_selection_supervised(train_X,train_Y,num_concepts,accuracies)

[0.24998083 0.24995266 0.24560361 0.24473498 0.24395134 0.24265612
 0.24214729 0.241924   0.2411571  0.2411571  0.23769547 0.2339816
 0.23339578 0.2330718  0.23246971 0.22802237 0.22429046 0.22422355
 0.2238877  0.21352187 0.20414544 0.2031579  0.19966545 0.19853649
 0.19691569 0.19340045 0.18822764 0.18770833 0.18486649 0.18390512
 0.17921471 0.17686562 0.17527825 0.1747072  0.17390409 0.17111819
 0.17029621 0.16672516 0.16282978 0.16270661 0.15972456 0.15922268
 0.15244146 0.15205032 0.15152757 0.14568555 0.14487586 0.13981365
 0.13702776 0.13660688 0.1354807  0.12947695 0.12787919 0.12612418
 0.12612418 0.12597737 0.12597737 0.12287415 0.1224277  0.12168189
 0.11837447 0.11209834 0.11147821 0.1110122  0.1110122  0.10835669
 0.10835669 0.10757088 0.10646708 0.10615093 0.10504166 0.10152765
 0.10120611 0.10056198 0.10040073 0.09845895 0.09666799 0.09535887
 0.09173008 0.08451593 0.08298565 0.08110574 0.07852527 0.07800682
 0.07766076 0.07731435 0.07731435 0.07627303 0.07609917 0.07313

In [ ]:
def get_performance_real(selected_concepts):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
    )

    # Train the model
    mlp.fit(pred_train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(intervention_test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

In [ ]:
results['intervention'] = {}

In [30]:
for intervention_percent in [0.2,0.4,0.6,0.8,1.0]:
    num_cols = test_X.shape[1]
    num_cols_to_intervene = int((1-intervention_percent) * num_cols)

    # Randomly pick columns
    cols = np.random.choice(
        num_cols, num_cols_to_intervene, replace=False
    )

    # Start from original
    intervention_test_X = test_X.copy()

    # Replace selected columns entirely
    intervention_test_X[:, cols] = pred_test_X[:, cols]

    for arr,description in zip([manually_selected_concepts,
                                lp_selection,
                                multiple_selection,
                                greedy_selection,
                                random_selection,
                                imperfect_selection],[
                                    "manual","lp","multiple",
                                    'greedy','random',
                                    'imperfect'
                                ]):
        if description not in results['intervention']:
            results['intervention'][description] = {}
        results['intervention'][description][intervention_percent] = {
            'reward': get_performance_real(arr)
        }
        print(description,intervention_percent,results['intervention'][description][intervention_percent]['reward'])


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


manual 0.2 0.9287193648602002


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


lp 0.2 0.8950638591646531


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


...

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


greedy 1.0 0.6139109423541594


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


random 1.0 0.6002761477390404
imperfect 1.0 0.5843976527442182


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


## Save Data

In [38]:
save_path = get_save_path(out_folder,save_name)

In [39]:
delete_duplicate_results(out_folder,"",results)

An error occurred: Expected object or value
An error occurred: Expected object or value
An error occurred: Expected object or value
An error occurred: Expected object or value


In [40]:
json.dump(results,open('../../results/'+save_path,'w'))